

We’ll (re)load all required data:
- `kinexon_positions` (all sessions) with a computed UTC `ts` column
- `kinexon_events_detected_enriched` (for optional initial detected times)
- `match_events` goals (review targets)

We also define **simple, explicit heuristics** for manual throw-point detection:
- **Possession** while `dist(player, ball) <= D_POSSESS` **and** `ball_speed <= V_POSSESS_MAX`
- **Release** = **first local maximum** of **ball acceleration** *after* the **last possession contact** (within a search window)
  - Fallback: first frame after last possession where `ball_speed >= V_RELEASE_MIN` and `dist >= D_RELEASE_MIN`

All timings are **absolute**, based on `ts in ms`.
"""


In [ ]:
# Cell 1 — Setup, data loading, and parameters

import numpy as np
import pandas as pd
from IPython.display import display

# Reuse existing connection `con` if already created; otherwise connect.
try:
    con
except NameError:
    import duckdb
    DB_PATH = "../data/mydb2024-25.duckdb"
    con = duckdb.connect(database=DB_PATH, read_only=False)
    print("Connected to DuckDB:", DB_PATH)

# select 5 distinct session ids
distinct_list_session_ids = con.execute("""
    SELECT DISTINCT session_id
    FROM kinexon_positions
    WHERE session_id IS NOT NULL
    LIMIT 5
""").fetchdf()
print("Distinct session IDs selected for analysis:")
display(distinct_list_session_ids)


# --- Load tables ---
df_positions_all = con.execute("""
    SELECT *
    FROM kinexon_positions
    WHERE session_id IN (""" + ",".join(f"'{sid}'" for sid in distinct_list_session_ids["session_id"]) + """)
""").fetchdf()
if "ts" not in df_positions_all.columns:
    df_positions_all["ts"] = pd.to_datetime(df_positions_all["ts in ms"], unit="ms", utc=True, errors="coerce")

df_enriched = con.execute("""
    SELECT *
    FROM kinexon_events_detected_enriched
""").fetchdf()

df_goals = con.execute("""
    SELECT fixtureId, eventId, eventTime, eventType, personId, goalKeeperId
    FROM match_events
    WHERE eventType = 'goal'
""").fetchdf()
df_goals["eventTime"] = pd.to_datetime(df_goals["eventTime"], utc=True, errors="coerce")
df_goals["eventTime_ms"] = (df_goals["eventTime"].astype("int64") // 10**6)

# Map shooter/goalkeeper league_id (for player filtering)
players_core = con.execute("""
    SELECT personId, league_id, nameFullLatin AS person_name, teamName
    FROM players
""").fetchdf()
df_goals = df_goals.merge(
    players_core.rename(columns={"league_id": "person_league_id"}),
    on="personId", how="left"
).merge(
    players_core.rename(columns={"personId": "goalKeeperId", "league_id": "goalkeeper_league_id"}),
    on="goalKeeperId", how="left"
)

# Optional: merge best detected kinexon timestamp from enriched (nearest match linkage)
m = (
    df_enriched
    .dropna(subset=["match_eventId", "match_time_diff_ms"])
    .assign(abs_dt=lambda d: d["match_time_diff_ms"].abs())
    .sort_values(["fixture_id", "match_eventId", "abs_dt"])
    .groupby(["fixture_id", "match_eventId"], as_index=False)
    .first()[["fixture_id", "match_eventId", "timestamp_ms", "match_time_diff_ms"]]
    .rename(columns={
        "fixture_id": "fixtureId",
        "match_eventId": "eventId",
        "timestamp_ms": "kinexon_matched_ts_ms"
    })
)
df_goals = df_goals.merge(m, on=["fixtureId","eventId"], how="left")
df_goals["kinexon_matched_ts"] = pd.to_datetime(df_goals["kinexon_matched_ts_ms"], unit="ms", utc=True, errors="coerce")

# session lookup per fixture
fx_sessions = con.execute("""
    SELECT DISTINCT fixture_id, ANY_VALUE(session_id) AS session_id
    FROM kinexon_positions
    WHERE fixture_id IS NOT NULL
    GROUP BY fixture_id
""").fetchdf()
fixture_to_session = dict(zip(fx_sessions["fixture_id"], fx_sessions["session_id"]))

print(f"Positions loaded: {len(df_positions_all):,}")
print(f"Enriched events loaded: {len(df_enriched):,}")
print(f"Goal rows loaded: {len(df_goals):,}")
display(df_goals.head(3))

# --- Heuristic parameters (tune if needed) ---
WINDOW_BEFORE_S = 2.5   # seconds before seed time to start scanning
WINDOW_AFTER_S  = 2.0   # seconds after seed time to end scanning
D_POSSESS = 2         # meters: player-ball distance to be considered possession
V_POSSESS_MAX = 6.0     # m/s: ball speed threshold during possession (usually low before release)
V_RELEASE_MIN = 9.0     # m/s: ball speed threshold just after release (handball shot ~ > 15 m/s; keep conservative)
D_RELEASE_MIN = 2.0     # meters: distance just after release grows beyond this
ACC_PEAK_NEIGH = 2      # frames neighborhood to qualify a local acceleration maximum
MIN_SAMPLES = 4         # minimal frames required to attempt detection

print("Heuristic parameters set:")
print(dict(
    WINDOW_BEFORE_S=WINDOW_BEFORE_S, WINDOW_AFTER_S=WINDOW_AFTER_S,
    D_POSSESS=D_POSSESS, V_POSSESS_MAX=V_POSSESS_MAX,
    V_RELEASE_MIN=V_RELEASE_MIN, D_RELEASE_MIN=D_RELEASE_MIN,
    ACC_PEAK_NEIGH=ACC_PEAK_NEIGH, MIN_SAMPLES=MIN_SAMPLES
))


### Cell 2 — Utility functions

- `extract_tracks_window(...)`: slice positions around a **seed time** (use Kinexon matched ts if available, otherwise `eventTime`)
- `build_joined_player_ball(...)`: align shooter and ball by timestamp, compute distance, speed, acceleration, and simple derivatives
- `detect_throw_point(...)`: compute last possession window and find the **first local maximum** in ball acceleration after possession ends; fallback to speed+distance threshold
- `first_local_maximum(...)`: helper to detect local maxima with a small neighborhood

In [ ]:
# Cell 2 — Utility functions

from typing import Optional, Tuple, Dict

def seed_time_from_row(row: pd.Series) -> pd.Timestamp:
    """Choose the seed time: Kinexon matched if available; else eventTime."""
    if pd.notna(row.get("kinexon_matched_ts")):
        return row["kinexon_matched_ts"]
    return row["eventTime"]

def extract_tracks_window(df_pos: pd.DataFrame,
                          session_id: int,
                          center_ts: pd.Timestamp,
                          w_before_s: float,
                          w_after_s: float) -> pd.DataFrame:
    """Slice positions for a session around center_ts with given margins."""
    if pd.isna(center_ts):
        return pd.DataFrame()
    t0 = center_ts - pd.Timedelta(seconds=w_before_s)
    t1 = center_ts + pd.Timedelta(seconds=w_after_s)
    df = df_pos[df_pos["session_id"] == session_id]
    if df.empty:
        return df
    df = df[(df["ts"] >= t0) & (df["ts"] <= t1)].copy()
    df.sort_values("ts", inplace=True)
    return df

def build_joined_player_ball(df_scene: pd.DataFrame,
                             shooter_league_id: Optional[int]) -> pd.DataFrame:
    """
    Align shooter track with ball track on timestamps.
    - Shooter identified by `league id == shooter_league_id`
    - Ball identified by `league id` containing "ball" or "Ball"
    Compute:
      - dist_pb (player-ball distance)
      - ball_speed (from 'speed in m/s' if available)
      - ball_acc (from 'acceleration in m/s2' if available; else finite diff on speed)
    """
    if df_scene.empty or pd.isna(shooter_league_id):
        return pd.DataFrame()

    # Ball identification: league id contains "ball" or "Ball"
    ball_mask = df_scene["league id"].astype(str).str.contains("ball|Ball", case=False, na=False)
    ball = df_scene[ball_mask].copy()
    # remove ball rows from df_scene to avoid confusion
    df_scene = df_scene[~ball_mask]
    shooter = df_scene[df_scene["league id"].astype("Int64") == int(shooter_league_id)].copy()

    if ball.empty or shooter.empty:
        return pd.DataFrame()

    # keep necessary columns
    cols_keep = ["ts", "ts in ms", "x in m", "y in m", "speed in m/s", "acceleration in m/s2"]
    ball = ball[cols_keep].rename(columns={
        "x in m": "ball_x", "y in m": "ball_y",
        "speed in m/s": "ball_speed", "acceleration in m/s2": "ball_acc"
    })
    shooter = shooter[cols_keep].rename(columns={
        "x in m": "pl_x", "y in m": "pl_y",
        "speed in m/s": "pl_speed", "acceleration in m/s2": "pl_acc"
    })

    # align by exact ts; if data has small jitter, consider merge_asof on ts
    df = pd.merge(ball, shooter, on="ts", how="inner", suffixes=("", "_pl"))
    if df.empty:
        # fallback: asof (nearest within 30 ms)
        df = pd.merge_asof(
            ball.sort_values("ts"), shooter.sort_values("ts"),
            on="ts", direction="nearest", tolerance=pd.Timedelta(milliseconds=30)
        )

    if df.empty:
        return df

    # distance and robust speed/acc
    df["dist_pb"] = np.hypot(df["ball_x"] - df["pl_x"], df["ball_y"] - df["pl_y"])

    # If ball_acc is missing, compute from ball_speed with finite difference
    if "ball_acc" not in df or df["ball_acc"].isna().all():
        df = df.sort_values("ts").copy()
        dt = df["ts"].diff().dt.total_seconds().fillna(0)
        bs = df["ball_speed"].fillna(method="ffill").fillna(0)
        df["ball_acc"] = (bs - bs.shift(1)) / dt.replace(0, np.nan)
        df["ball_acc"].replace([np.inf, -np.inf], np.nan, inplace=True)

    return df

def first_local_maximum(values: np.ndarray, neigh: int = 2) -> Optional[int]:
    """
    Return index of the first local maximum with ±neigh neighborhood,
    ignoring NaNs. If none found, return None.
    """
    x = np.array(values, dtype=float)
    n = x.size
    if n == 0:
        return None
    for i in range(neigh, n - neigh):
        if np.isnan(x[i]):
            continue
        left = x[i - neigh:i]
        right = x[i + 1:i + 1 + neigh]
        if np.all(x[i] >= left) and np.all(x[i] > right):
            return i
    return None

def detect_throw_point(df_pb: pd.DataFrame,
                       d_possess: float,
                       v_possess_max: float,
                       v_release_min: float,
                       d_release_min: float,
                       acc_peak_neigh: int) -> Dict[str, Optional[object]]:
    """
    Compute possession and throw point:
      1) possession mask: (dist_pb <= d_possess) & (ball_speed <= v_possess_max)
      2) last_possession_idx: last index where possession == True
      3) search after last_possession_idx for local maximum in ball_acc → throw_idx
         - if no peak: fallback to first idx where (ball_speed >= v_release_min AND dist_pb >= d_release_min)
    Returns metadata dict with indices and timestamps.
    """
    out = dict(
        n_rows=int(len(df_pb)),
        last_possession_idx=None,
        throw_idx=None,
        throw_ts=None,
        method=None
    )
    if df_pb.empty or len(df_pb) < MIN_SAMPLES:
        return out

    poss = (df_pb["dist_pb"] <= d_possess) & (df_pb["ball_speed"].fillna(0) <= v_possess_max)
    if poss.any():
        last_possession_idx = poss[poss].index.max()
        out["last_possession_idx"] = int(df_pb.index.get_indexer_for([last_possession_idx])[0])
        # Now search after this index
        post = df_pb.loc[df_pb.index > last_possession_idx].copy()
    else:
        # No possession detected — still try from the beginning of the window
        out["last_possession_idx"] = None
        post = df_pb.copy()

    if not post.empty:
        # Local maxima in ball acceleration
        acc = post["ball_acc"].to_numpy(dtype=float)
        peak_rel = first_local_maximum(acc, neigh=acc_peak_neigh)
        if peak_rel is not None:
            throw_row = post.iloc[peak_rel]
            out["throw_idx"] = int(df_pb.index.get_indexer_for([throw_row.name])[0])
            out["throw_ts"]  = throw_row["ts"]
            out["method"] = "acc_peak"
            return out

        # Fallback: speed & distance threshold
        cond = (post["ball_speed"].fillna(0) >= v_release_min) & (post["dist_pb"] >= d_release_min)
        if cond.any():
            throw_row = post[cond].iloc[0]
            out["throw_idx"] = int(df_pb.index.get_indexer_for([throw_row.name])[0])
            out["throw_ts"]  = throw_row["ts"]
            out["method"] = "speed_dist_fallback"
            return out

    return out


In [ ]:
# --- Fancy plotting helper (human time axis HH:MM:SS.mmm, annotations, peaks) ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

def _ts_to_ms(series_or_ts):
    s = pd.to_datetime(series_or_ts, utc=True, errors="coerce")
    if isinstance(s, pd.Series):
        return (s.astype("int64") // 10**6)
    return None if pd.isna(s) else int(s.value // 10**6)

def _fmt_time_ms(ms_val, _pos=None):
    """Format milliseconds since epoch to HH:MM:SS.mmm (UTC)."""
    try:
        ts = pd.to_datetime(int(ms_val), unit="ms", utc=True)
        return ts.strftime("%H:%M:%S.%f")[:-3]
    except Exception:
        return ""

def _find_local_maxima(values: np.ndarray, neigh: int = 2) -> list[int]:
    x = np.asarray(values, dtype=float)
    n = x.size
    peaks = []
    if n == 0:
        return peaks
    for i in range(neigh, n - neigh):
        if np.isnan(x[i]):
            continue
        if np.all(x[i] >= x[i - neigh:i]) and np.all(x[i] > x[i + 1:i + 1 + neigh]):
            peaks.append(i)
    return peaks

def _annot(ax, x, y, text, color, dy=0.0):
    ax.annotate(
        text, xy=(x, y), xytext=(0, 12 + dy),
        textcoords="offset points", ha="center", va="bottom",
        fontsize=9, color=color,
        arrowprops=dict(arrowstyle="-", color=color, lw=1, alpha=0.8)
    )

def _plot_event_sync(df_pb: pd.DataFrame,
                     *,
                     last_possession_idx: int | None,
                     event_ms: int | None,
                     kin_ms: int | None,
                     throw_ms: int | None,
                     title: str | None = None):
    """
    Fancy twin-axis plot:
      - Left y: ball_acc (m/s²) with zero line and peak markers
      - Right y: player–ball distance (m) + optional ball speed (faint)
      - Shaded possession and post-possession search region
      - Vertical markers for event/kinexon/refined + small annotations
      - X axis as HH:MM:SS.mmm (UTC)
    """
    if df_pb.empty:
        return

    # Prepare data
    x_ms = _ts_to_ms(df_pb["ts"])
    acc  = df_pb["ball_acc"].to_numpy(dtype=float)
    dist = df_pb["dist_pb"].to_numpy(dtype=float)
    spd  = df_pb["ball_speed"].to_numpy(dtype=float)

    # Heuristic masks
    d_possess = globals().get("D_POSSESS", 1.2)
    v_possess = globals().get("V_POSSESS_MAX", 6.0)
    possess = (df_pb["dist_pb"] <= d_possess) & (df_pb["ball_speed"].fillna(0) <= v_possess)

    # Peaks: look after the last possession index if available
    neigh = globals().get("ACC_PEAK_NEIGH", 2)
    start_idx = (last_possession_idx + 1) if last_possession_idx is not None else 0
    acc_post = acc[start_idx:]
    rel_peaks = _find_local_maxima(acc_post, neigh=neigh)
    peaks_idx = [start_idx + rp for rp in rel_peaks]
    release_idx = peaks_idx[0] if len(peaks_idx) else None

    # Figure
    fig, axA = plt.subplots(figsize=(11.5, 5.2))
    axA.xaxis.set_major_formatter(FuncFormatter(_fmt_time_ms))

    # Left y: acceleration
    acc_line, = axA.plot(x_ms, acc, lw=1.6, label="Ball acceleration (m/s²)")
    axA.axhline(0, lw=1, color="0.7")
    axA.set_xlabel("Time (HH:MM:SS.mmm, UTC)")
    axA.set_ylabel("Acceleration (m/s²)")

    # Right y: distance (+ optional speed)
    axD = axA.twinx()
    dist_line, = axD.plot(x_ms, dist, ls="--", lw=1.6, label="Player–ball distance (m)")
    try:
        # Subtle speed overlay (can be useful to see the jump)
        spd_line, = axD.plot(x_ms, spd, ls="-.", lw=1.0, alpha=0.5, label="Ball speed (m/s)")
    except Exception:
        spd_line = None
    axD.set_ylabel("Distance / Speed")

    # Possession shading
    ymax = np.nanmax(dist) if np.isfinite(dist).any() else 20
    axD.fill_between(x_ms, 0, ymax, where=possess.to_numpy(),
                     alpha=0.18, color="#6aa84f", label="Possession")

    # Post-possession search shading
    if last_possession_idx is not None and last_possession_idx + 1 < len(x_ms):
        axA.axvspan(x_ms.iloc[last_possession_idx + 1], x_ms.iloc[-1],
                    alpha=0.08, color="#999999", label="Post-possession search")

    # Vertical markers and annotations
    markers = [
        (event_ms,   "Match eventTime",  "#1f77b4"),
        (kin_ms,     "Kinexon matched",  "#ff7f0e"),
        (throw_ms,   "Refined throw",    "#d62728"),
    ]
    for ms, lab, col in markers:
        if ms is None:
            continue
        axA.axvline(ms, color=col, ls=":", lw=1.8, label=lab)
        # annotate near acceleration curve at its local value (or zero)
        idx = int(np.clip(np.searchsorted(x_ms.to_numpy(), ms), 0, len(acc) - 1))
        y_here = np.nan_to_num(acc[idx], nan=0.0)
        _annot(axA, ms, y_here, lab, col)

    # Peak markers
    if len(peaks_idx):
        axA.scatter(x_ms.iloc[peaks_idx], acc[peaks_idx], s=28, zorder=3,
                    label="Acc local maxima")
    if release_idx is not None:
        axA.axvline(x_ms.iloc[release_idx], color="#d62728", ls="--", lw=2.0, label="First peak (release)")

    # Title & legend
    if title:
        axA.set_title(title)
    lines, labels = axA.get_legend_handles_labels()
    r_lines, r_labels = axD.get_legend_handles_labels()
    # Remove duplicates while preserving order
    seen = set()
    L, S = [], []
    for h, t in list(zip(lines + r_lines, labels + r_labels)):
        if t and t not in seen:
            seen.add(t); L.append(h); S.append(t)
    axA.legend(L, S, loc="upper left", ncol=2, frameon=True)

    axA.grid(True, alpha=0.28)
    fig.tight_layout()
    plt.show()


### Cell 3 — Per-event refinement function

`refine_throw_time_for_event(row)`:
- Locate session for the event’s fixture.
- Slice a **tight window** around the seed time (`kinexon_matched_ts` if present, else `eventTime`).
- Build aligned player–ball series, compute distance/speed/acceleration.
- Detect **throw point** using the heuristic.
- Return a **diagnostics row** with timestamps (ms), deltas, and method used.

In [ ]:
# Cell 3 — Per-event refinement function (with optional plotting)

from typing import Optional, Tuple, Dict

def refine_throw_time_for_event(row: pd.Series,
                                df_positions_all: pd.DataFrame,
                                *,
                                plot: bool = False,
                                plot_title: str | None = None) -> Optional[pd.Series]:
    fixture_id = row["fixtureId"]
    session_id = fixture_to_session.get(fixture_id)
    shooter_league_id = row.get("person_league_id")
    seed_ts = seed_time_from_row(row)

    diag = dict(
        fixtureId=fixture_id,
        eventId=row["eventId"],
        seed_ts=seed_ts,
        seed_ms=(seed_ts.value // 10**6) if pd.notna(seed_ts) else None,
        eventTime=row.get("eventTime"),
        eventTime_ms=row.get("eventTime_ms"),
        kinexon_matched_ts=row.get("kinexon_matched_ts"),
        kinexon_matched_ts_ms=row.get("kinexon_matched_ts_ms"),
        refined_throw_ts=None,
        refined_throw_ts_ms=None,
        refined_delta_vs_event_ms=None,
        refined_delta_vs_seed_ms=None,
        method=None,
        n_rows_window=None,
        last_possession_idx=None
    )

    if session_id is None or pd.isna(shooter_league_id) or pd.isna(seed_ts):
        return pd.Series(diag)

    # window slice
    df_scene = extract_tracks_window(
        df_positions_all, session_id, seed_ts,
        w_before_s=WINDOW_BEFORE_S, w_after_s=WINDOW_AFTER_S
    )
    diag["n_rows_window"] = int(len(df_scene))

    if df_scene.empty:
        return pd.Series(diag)

    df_pb = build_joined_player_ball(df_scene, shooter_league_id)
    if df_pb.empty:
        return pd.Series(diag)

    # Ensure index is simple increasing for index math
    df_pb = df_pb.reset_index(drop=True)

    result = detect_throw_point(
        df_pb,
        d_possess=D_POSSESS,
        v_possess_max=V_POSSESS_MAX,
        v_release_min=V_RELEASE_MIN,
        d_release_min=D_RELEASE_MIN,
        acc_peak_neigh=ACC_PEAK_NEIGH
    )

    # Attach results
    diag["last_possession_idx"] = result.get("last_possession_idx")
    if result.get("throw_ts") is not None:
        t = pd.to_datetime(result["throw_ts"], utc=True)
        diag["refined_throw_ts"] = t
        diag["refined_throw_ts_ms"] = int(t.value // 10**6)
        diag["method"] = result.get("method")

        if pd.notna(diag["eventTime_ms"]):
            diag["refined_delta_vs_event_ms"] = int(diag["refined_throw_ts_ms"] - int(diag["eventTime_ms"]))
        if diag["seed_ms"] is not None:
            diag["refined_delta_vs_seed_ms"]  = int(diag["refined_throw_ts_ms"] - int(diag["seed_ms"]))

    # --- Optional plotting: one line toggle ---
    if plot:
        _plot_event_sync(
            df_pb,
            last_possession_idx=diag["last_possession_idx"],
            event_ms=int(diag["eventTime_ms"]) if pd.notna(diag["eventTime_ms"]) else None,
            kin_ms=int(diag["kinexon_matched_ts_ms"]) if pd.notna(diag["kinexon_matched_ts_ms"]) else None,
            throw_ms=int(diag["refined_throw_ts_ms"]) if diag["refined_throw_ts_ms"] is not None else None,
            title=plot_title or f"Event sync — {row.get('person_name', '')} ({row.get('eventId','')})"
        )

    return pd.Series(diag)


### Cell 4 — Run refinement over all goal events, print progress, inspect results

- Applies the refinement to **all** goals.
- Displays:
  - head of the diagnostics
  - basic stats of deltas vs event time and seed time
  - a small sample where method is `acc_peak` and another for fallback

In [ ]:
# Cell 4 — Run refinement over all goal events, print progress, inspect results

diagnostics = []
for i, row in df_goals.iterrows():
    s = refine_throw_time_for_event(row, df_positions_all, plot=True)
    diagnostics.append(s)
    if (i+1) % 20 == 0:
        print(f"Processed {i+1}/{len(df_goals)}")
        break

df_refined = pd.DataFrame(diagnostics)
print(f"Refined {len(df_refined)} goal events.")
display(df_refined.head(8))

# Stats
def describe_series(name, s):
    s_num = pd.to_numeric(s, errors="coerce").dropna()
    if s_num.empty:
        print(f"{name}: no data")
        return
    print(f"{name}: count={len(s_num)}, mean={s_num.mean():.1f} ms, median={s_num.median():.1f} ms, "
          f"min={s_num.min():.1f}, max={s_num.max():.1f}")

describe_series("Δ refined vs eventTime (ms)", df_refined["refined_delta_vs_event_ms"])
describe_series("Δ refined vs seed (ms)", df_refined["refined_delta_vs_seed_ms"])

print("\nSample — acc_peak detections:")
display(df_refined[df_refined["method"] == "acc_peak"].head(5))

print("\nSample — fallback detections:")
display(df_refined[df_refined["method"] == "speed_dist_fallback"].head(5))


### Cell 5 — Persist refined throw-points to DuckDB

We save to `kinexon_throwpoint_refined`:
- `fixtureId`, `eventId`
- `refined_throw_ts` and `refined_throw_ts_ms`
- deltas vs. event and seed
- detection `method`
- a few references (`seed_ts`, `eventTime`, `kinexon_matched_ts`

In [ ]:
# Cell 5 — Persist refined throw-points to DuckDB

TABLE = "kinexon_throwpoint_refined"

con.execute(f"DROP TABLE IF EXISTS {TABLE}")
con.register("df", df_refined)
con.execute(f"""
    CREATE TABLE {TABLE} AS
    SELECT
        fixtureId,
        eventId,
        eventTime,
        eventTime_ms,
        seed_ts,
        seed_ms,
        kinexon_matched_ts,
        kinexon_matched_ts_ms,
        refined_throw_ts,
        refined_throw_ts_ms,
        refined_delta_vs_event_ms,
        refined_delta_vs_seed_ms,
        method,
        n_rows_window,
        last_possession_idx
    FROM df
""")
con.unregister("df")

written = con.execute(f"SELECT COUNT(*) FROM {TABLE}").fetchone()[0]
print(f"✓ Wrote {written} rows into {TABLE}")

display(con.execute(f"SELECT * FROM {TABLE} LIMIT 10").fetchdf())


### Cell 6 — Inspect a single event end-to-end (debug printout)

For a chosen `eventId`, show:
- the chosen **seed time**
- first/last timestamps in the analysis window
- whether a possession window existed
- the detected throw time and method
- a small excerpt of the aligned player–ball table around the detection

In [ ]:
# Cell 6 — Inspect a single event end-to-end (debug printout)

EVENT_ID_INSPECT = None  # e.g. "b7d8cd90-8ed2-11ef-b16d-cda25f1166cf"
# get an eventId from df_goals to inspect
EVENT_ID_INSPECT = df_goals["eventId"].iloc[0] if not df_goals.empty else None

if EVENT_ID_INSPECT:
    g = df_goals[df_goals["eventId"] == EVENT_ID_INSPECT]
    if g.empty:
        print("EventId not found.")
    else:
        row = g.iloc[0]
        fixture_id = row["fixtureId"]
        session_id = fixture_to_session.get(fixture_id)
        seed_ts = seed_time_from_row(row)

        print("Fixture:", fixture_id)
        print("Session:", session_id)
        print("Seed time:", seed_ts)

        df_scene = extract_tracks_window(
            df_positions_all, session_id, seed_ts,
            w_before_s=WINDOW_BEFORE_S, w_after_s=WINDOW_AFTER_S
        )
        if df_scene.empty:
            print("No frames in window.")
        else:
            print("Window ts range:", df_scene["ts"].min(), "→", df_scene["ts"].max())
            df_pb = build_joined_player_ball(df_scene, row.get("person_league_id"))
            if df_pb.empty:
                print("No aligned player–ball rows.")
            else:
                res = detect_throw_point(
                    df_pb,
                    d_possess=D_POSSESS,
                    v_possess_max=V_POSSESS_MAX,
                    v_release_min=V_RELEASE_MIN,
                    d_release_min=D_RELEASE_MIN,
                    acc_peak_neigh=ACC_PEAK_NEIGH
                )
                print("Detection method:", res.get("method"))
                print("Throw ts:", res.get("throw_ts"))
                print("Last possession idx:", res.get("last_possession_idx"))

                # Show a small excerpt around the detected point
                if res.get("throw_ts") is not None:
                    t = pd.to_datetime(res["throw_ts"], utc=True)
                    around = df_pb[
                        (df_pb["ts"] >= t - pd.Timedelta(milliseconds=150)) &
                        (df_pb["ts"] <= t + pd.Timedelta(milliseconds=150))
                    ][["ts","dist_pb","ball_speed","ball_acc","pl_x","pl_y","ball_x","ball_y"]]
                    print("\nExcerpt around throw:")
                    display(around)
                else:
                    display(df_pb.head(8))


In [ ]:
# Cell 13 — Build freeze candidates & robust display fields

from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

# 1) Refined throwpoints
df_refined_tp = con.execute("""
    SELECT fixtureId, eventId, refined_throw_ts
    FROM kinexon_throwpoint_refined
""").fetchdf()
df_refined_tp["refined_throw_ts"] = pd.to_datetime(df_refined_tp["refined_throw_ts"], utc=True, errors="coerce")

# 2) Merge onto goals collected earlier (df_goals already has person_league_id / goalkeeper_league_id and merges from players)
goals_all = df_goals.merge(df_refined_tp, on=["fixtureId", "eventId"], how="left")

# 3) Derive robust display fields
def _coalesce(series_list):
    for s in series_list:
        if s is not None and s.name in goals_all.columns:
            return s
    # if nothing found, return a NaN series of the right length
    return pd.Series([np.nan] * len(goals_all), index=goals_all.index)

goals_all["teamName_disp"] = _coalesce([
    goals_all.get("teamName"),            # from match_events, if selected in Cell 1
    goals_all.get("teamName_x"),          # sometimes appears after merges
    goals_all.get("teamName_y"),          # fallback
    goals_all.get("teamName_players"),    # if you renamed from players explicitly
    goals_all.get("teamName")             # final fallback
])

goals_all["personName_disp"] = _coalesce([
    goals_all.get("personName"),          # from match_events, if selected
    goals_all.get("person_name"),         # from players merge (nameFullLatin AS person_name)
])

goals_all["goalkeeperName_disp"] = _coalesce([
    goals_all.get("goalkeeperName"),          # from match_events, if selected
    goals_all.get("goalkeeper_person_name"),  # from players merge (second join)
])

# 4) Preview (only show columns that truly exist)
summary_cols_requested = [
    "fixtureId", "eventId", "eventType",
    "teamName_disp", "personName_disp", "goalkeeperName_disp",
    "eventTime", "kinexon_matched_ts", "refined_throw_ts"
]
summary_cols = [c for c in summary_cols_requested if c in goals_all.columns]
print(f"Prepared goals with freeze candidates: {len(goals_all)}")
display(goals_all[summary_cols].head(5))

# 5) Ensure rendering constants/paths exist in this notebook
FPS = 60
FRAME_INTERVAL_MS = int(round(1000 / FPS))
POS_PAD_SEC_BEFORE = 15
POS_PAD_SEC_AFTER  =  3
TRAIL_FRAMES = 5

# Freeze durations (seconds)
FREEZE_SECONDS_EVENT  = 1.5
FREEZE_SECONDS_KIN    = 1.5
FREEZE_SECONDS_MANUAL = 1.5

# Colors (BGR for OpenCV)
COLOR_EVENT  = (255, 0, 0)     # Blue
COLOR_KIN    = (0, 255, 255)   # Yellow
COLOR_MANUAL = (255, 0, 255)   # Magenta

# Field image / output dir from earlier cells, else set sensible defaults
from pathlib import Path
if "FIELD_IMAGE" not in globals():
    FIELD_IMAGE = Path("../data/metadata/handballfeld.png")
if "OUT_DIR" not in globals():
    OUT_DIR = Path("../data/metadata")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Freeze configuration (seconds):",
      dict(event=FREEZE_SECONDS_EVENT, kinexon=FREEZE_SECONDS_KIN, manual=FREEZE_SECONDS_MANUAL))
print("Paths:", {"FIELD_IMAGE": str(FIELD_IMAGE), "OUT_DIR": str(OUT_DIR)})

# Optional: quick introspection to help future mismatches
print("\n[Debug] Available columns in goals_all:")
print(sorted(goals_all.columns.tolist())[:30], "…")
print("…", sorted(goals_all.columns.tolist())[-30:])


In [ ]:
# Cell 14 — Multi-freeze renderer

import cv2
import numpy as np
from pathlib import Path

def render_goal_with_multifreeze(
    df_positions: pd.DataFrame,
    row_goal: pd.Series,
    freeze_markers: list,               # list of dicts: {"name","ts","color","seconds"}
    field_image_path: Path = None,
    out_dir: Path = None,
    fps: int = None,
) -> Path | None:
    """
    Render a goal clip and freeze at multiple marker times.
    Each marker has its own label and border color.
    """
    # Defaults from notebook globals
    field_image_path = field_image_path or FIELD_IMAGE
    out_dir = Path(out_dir or OUT_DIR)
    fps = fps or FPS
    frame_interval_ms = int(round(1000 / fps))

    img = cv2.imread(str(field_image_path))
    if img is None:
        print(f"⚠️ Could not read field image at: {field_image_path}")
        return None

    height, width = img.shape[:2]
    scale = width / 40.0

    # Keep markers with valid timestamps
    valid_markers = [m for m in freeze_markers if pd.notna(m.get("ts"))]
    if not valid_markers:
        print("⚠️ No valid freeze markers for this event, skipping.")
        return None

    # Clip window that includes all markers
    t_min = min(m["ts"] for m in valid_markers)
    t_max = max(m["ts"] for m in valid_markers)
    t0 = t_min - pd.Timedelta(seconds=POS_PAD_SEC_BEFORE)
    t1 = t_max + pd.Timedelta(seconds=POS_PAD_SEC_AFTER)

    # Slice positions for this fixture/session & window
    df_scene = df_positions[(df_positions["ts"] >= t0) & (df_positions["ts"] <= t1)].copy()
    if df_scene.empty:
        print("⚠️ No positional data in window for eventId:", row_goal["eventId"])
        return None
    df_scene = df_scene.sort_values("ts").copy()

    # Frame indexing & ball arrow support
    df_scene["frame_idx"] = df_scene.groupby("ts").ngroup()
    df_scene["prev_x"] = df_scene.groupby(["mapped id"])["x in m"].shift(1)
    df_scene["prev_y"] = df_scene.groupby(["mapped id"])["y in m"].shift(1)

    # Output
    name_event = f'event_{row_goal["eventId"]}'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{name_event}.mp4"
    out_path_img = out_dir / f"{name_event}.png"
    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # Prepare marker state
    for m in valid_markers:
        m["ms"] = int(pd.Timestamp(m["ts"]).value // 10**6)
        m["frames"] = int(round(m["seconds"] * fps))
        m["done"] = False

    shooter_league_id = row_goal.get("person_league_id", None)
    goalkeeper_league_id = row_goal.get("goalkeeper_league_id", None)

    def color_for_group(group_id: float):
        # 3: ball, 2/1: teams
        if group_id == 3:  return (0, 0, 255), 15
        if group_id == 2:  return (0, 200, 0), 10
        if group_id == 1:  return (255, 140, 0), 10
        return (200, 200, 200), 8

    def draw_overlay(img_draw, ts):
        hud_lines = [
            f"Fixture: {row_goal.get('fixtureId','')}",
            f"EventId: {row_goal.get('eventId','')}",
            f"Shooter ID: {row_goal.get('person_league_id','N/A')}  |  GK ID: {row_goal.get('goalkeeper_league_id','N/A')}",
            f"Frame time (UTC): {ts.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}",
            "Markers: " + ", ".join([m['name'] for m in valid_markers]),
        ]
        y0 = 28
        for line in hud_lines:
            cv2.putText(img_draw, line, (10, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 2, cv2.LINE_AA)
            y0 += 24

    first_png_saved = False

    for ts, group in df_scene.groupby("ts"):
        img_draw = img.copy()

        # Draw actors
        for _, r in group.iterrows():
            gid = r.get("group id", None)
            color, radius = color_for_group(gid)
            if pd.isna(r["x in m"]) or pd.isna(r["y in m"]):
                continue
            x = int(float(r["x in m"]) * scale)
            y = int(float(r["y in m"]) * scale)
            cv2.circle(img_draw, (x, y), radius, color, -1, lineType=cv2.LINE_AA)

            # Labels & halos
            league_id = r.get("league id", "N/A")
            name = r.get("full name", "N/A")
            cv2.putText(img_draw, f"{name}", (x - 12, y - radius - 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.40, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(img_draw, f"ID:{league_id}", (x - 12, y - radius - 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.40, (255, 255, 255), 1, cv2.LINE_AA)
            try:
                if shooter_league_id is not None and int(r.get("league id", -1)) == int(shooter_league_id):
                    cv2.circle(img_draw, (x, y), radius + 6, (255, 255, 0), 2, cv2.LINE_AA)
                if goalkeeper_league_id is not None and int(r.get("league id", -1)) == int(goalkeeper_league_id):
                    cv2.circle(img_draw, (x, y), radius + 6, (0, 255, 255), 2, cv2.LINE_AA)
            except Exception:
                pass

        # Trails
        current_idx = group["frame_idx"].iloc[0]
        if TRAIL_FRAMES > 0:
            trail_slice = df_scene[
                (df_scene["frame_idx"] <= current_idx) &
                (df_scene["frame_idx"] > current_idx - TRAIL_FRAMES)
            ]
            for _, r in trail_slice.iterrows():
                if pd.isna(r["x in m"]) or pd.isna(r["y in m"]):
                    continue
                x_t = int(float(r["x in m"]) * scale)
                y_t = int(float(r["y in m"]) * scale)
                gid_t = r.get("group id", None)
                c_t, _ = color_for_group(gid_t)
                cv2.circle(img_draw, (x_t, y_t), 3, c_t, -1, cv2.LINE_AA)

        # Ball velocity arrow
        for _, br in group[group["group id"] == 3].iterrows():
            if not (pd.isna(br["prev_x"]) or pd.isna(br["prev_y"])):
                x0, y0 = int(br["prev_x"] * scale), int(br["prev_y"] * scale)
                x1, y1 = int(br["x in m"] * scale), int(br["y in m"] * scale)
                cv2.arrowedLine(img_draw, (x0, y0), (x1, y1), (50,50,255), 2, tipLength=0.3)

        # HUD
        draw_overlay(img_draw, ts)

        # Freezes
        ts_ms = int(ts.value // 10**6)
        any_frozen = False
        for m in valid_markers:
            if m["done"]:
                continue
            if abs(ts_ms - m["ms"]) <= (frame_interval_ms // 2):
                # Border + label
                cv2.rectangle(img_draw, (0,0), (width-1,height-1), m["color"], 6, cv2.LINE_AA)
                cv2.putText(img_draw, m["name"], (width//2 - 160, 80),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, m["color"], 2, cv2.LINE_AA)

                if not first_png_saved:
                    cv2.imwrite(str(out_path_img), img_draw)
                    first_png_saved = True

                # Write same frame 'frames' times for freeze effect
                for _ in range(m["frames"]):
                    cv2.imshow("Render", img_draw)
                    cv2.waitKey(1)
                    writer.write(cv2.resize(img_draw, (width, height)))
                m["done"] = True
                any_frozen = True
        if any_frozen:
            continue

        if not first_png_saved:
            cv2.imwrite(str(out_path_img), img_draw)
            first_png_saved = True

        cv2.imshow("Render", img_draw)
        cv2.waitKey(1)

        writer.write(cv2.resize(img_draw, (width, height)))

    writer.release()
    print(f"🎬 Saved: {out_path.name}  | 🖼️ Preview: {out_path_img.name}")
    return out_path


In [ ]:
# Cell 15 — Run rendering with multiple freeze points

RENDER_LIMIT = 50   # set None for all
rendered = 0

for _, row in goals_all.iterrows():
    fixture_id = row["fixtureId"]
    session_id = row["session_id"]
    if session_id is None:
        print(f"⚠️ No session_id for fixture {fixture_id}, skipping")
        continue

    # Narrow positions to this match/session
    df_pos_fx = df_positions_all[df_positions_all["session_id"] == session_id].copy()
    if df_pos_fx.empty:
        print(f"⚠️ No positions for session {session_id}, skipping")
        continue

    # Build freeze markers (only if timestamp exists)
    markers = []
    if pd.notna(row.get("eventTime")):
        markers.append({
            "name": "EVENT TIME",
            "ts": pd.to_datetime(row["eventTime"], utc=True),
            "color": COLOR_EVENT,
            "seconds": FREEZE_SECONDS_EVENT
        })
    if pd.notna(row.get("kinexon_matched_ts")):
        markers.append({
            "name": "KINEXON DETECTED",
            "ts": pd.to_datetime(row["kinexon_matched_ts"], utc=True),
            "color": COLOR_KIN,
            "seconds": FREEZE_SECONDS_KIN
        })
    if pd.notna(row.get("refined_throw_ts")):
        markers.append({
            "name": "MANUAL REFINED",
            "ts": pd.to_datetime(row["refined_throw_ts"], utc=True),
            "color": COLOR_MANUAL,
            "seconds": FREEZE_SECONDS_MANUAL
        })

    if not markers:
        print(f"No markers for eventId={row['eventId']}, skipping")
        continue

    # Display the plan (one row)
    display(pd.DataFrame([{
        "fixtureId": fixture_id,
        "eventId": row["eventId"],
        **{m["name"]: m["ts"] for m in markers}
    }]))

    # Render
    render_goal_with_multifreeze(df_pos_fx, row, markers)

    rendered += 1
    if RENDER_LIMIT and rendered >= RENDER_LIMIT:
        break

print(f"Done. Rendered {rendered} event(s) with multi-freeze.")
